## Structured Output

#### Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model('gemini-2.5-flash', model_provider='google_genai')
model.invoke('Hi')

/home/saddar/gen_ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c0dc8-e49d-7792-af7f-fd111158a91f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 44, 'total_tokens': 46, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 34}})

## Pydantic

#### Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="Title of the movie.")
    year:int = Field(description="This is the year the movie was released.")
    director:str = Field(description="The director of the movie.")
    rating: float = Field(description="The movies rating out of 10.")

In [5]:
structured_output_model = model.with_structured_output(Movie)
structured_output_model

RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x7ed76f1c3620>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'description': 'Title of the movie.', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'This is the year the movie was released.', 'title': 'Year', 'type': 'integer'}, 'director': {'description': 'The director of the movie.', 'title': 'Director', 'type': 'string'}, 'rating': {'description': 'The movi

In [6]:
response = structured_output_model.invoke("Provide me the details of the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

#### Raw message along with parsed structure

#### This is helpful when for any reason the output is not parsed you can always see raw message and find reason of not parsing the output

In [7]:
class Movie(BaseModel):
    """A movie with details"""
    title:str = Field(...,description='title of the movie')
    year:int = Field(...,description='the year in which movie was released')
    director:str = Field(...,description='director of the movie')
    rating:float = Field(...,description='The rating of the movie out of 10')

structured_output_model = model.with_structured_output(Movie, include_raw=True)
response = structured_output_model.invoke("Provide details about the movie Inception.")
response

{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c0957-97d5-7741-837c-de02b5cab944-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 109, 'total_tokens': 118, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 85}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

### Nested Structure

In [8]:
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor] # Nested Structure as Actor class is used inside MovieDetails class
    genres:list[str]
    budget:float | None = Field(None, description="Budget in millions USD.")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Please provide the details of the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dominick "Dom" Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Mr. Saito'), Actor(name='Cillian Murphy', role='Robert Michael Fischer')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

## TypedDict

#### TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [4]:
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]

In [5]:
model_with_typeddict = model.with_structured_output(Movie)
response = model_with_typeddict.invoke("Provide details about the movie Avengers Endgame.")
response

{'title': 'Avengers: Endgame',
 'year': 2019,
 'director': 'Anthony and Joe Russo',
 'rating': 8.4}

## Data Classes

#### Data classes provide a simple way to define a class with only data fields. They are ideal when you don't need any additional functionality. You create it using the dataclass decorator. @dataclass

In [6]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContantInfo:
    """Contact information of person"""
    name:str
    email:str
    phone:str

agent = create_agent(
    model=model,
    response_format=ContantInfo
)
response = agent.invoke({
    "messages":[{'role':'user','content':'Extract contact information from: John Doe, john.doe@example.com, 123-456-7890'}]
})    
response

{'messages': [HumanMessage(content='Extract contact information from: John Doe, john.doe@example.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='9a9fc363-9dcd-40e2-8d37-a871cb4a3a2a'),
  AIMessage(content='{"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890"}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c0dd8-1b63-7d72-977c-c36ac24277d2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 242, 'total_tokens': 272, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 211}})],
 'structured_response': ContantInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')}

In [7]:
response['structured_response']

ContantInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')